# LLM Cluster Naming

Questo notebook prende le keyword estratte dai diversi algoritmi (YAKE, TextRank, c-TF-IDF, KeyBERT) per ogni cluster e le passa a un modello LLM locale (Ollama - Qwen) per generare un nome sintetico di 1-3 parole.

In [1]:
import json
import os
import sys
import pandas as pd
from tqdm.notebook import tqdm

sys.path.append(os.path.abspath("../../"))
from src.utils.llm_naming import get_llm_cluster_name

In [2]:
# Caricamento metadata attuali
metadata_path = "../../data/metadata/cluster_labeling_metadata.json"
with open(metadata_path, "r") as f:
    metadata = json.load(f)

cluster_labels = metadata.get("cluster_labels", {})
print(f"Trovati {len(cluster_labels)} cluster da rinominare.")

Trovati 13 cluster da rinominare.


In [3]:
# Chiamate all'LLM
for cid, labels in tqdm(cluster_labels.items()):
    # Raccogliamo tutte le keyword dai vari algoritmi
    all_keywords = []
    all_keywords.extend(labels.get("ctfidf", []))
    all_keywords.extend(labels.get("yake", []))
    all_keywords.extend(labels.get("keybert_sim", []))
    # Rimuoviamo duplicati mantenendo l'ordine per importanza
    unique_keywords = list(dict.fromkeys(all_keywords))
    
    # Limitiamo a 20 (come da task)
    top_20 = unique_keywords[:20]
    
    # Chiamata LLM al modello locale Qwen3.5
    llm_name = get_llm_cluster_name(top_20, model="llama3")
    
    # Salviamo nel dizionario
    cluster_labels[cid]["llm_summary"] = llm_name

metadata["cluster_labels"] = cluster_labels

# Salviamo i risultati aggiornati
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=4)
print("Naming completato e file aggiornato!")

  0%|          | 0/13 [00:00<?, ?it/s]

Naming completato e file aggiornato!


In [4]:
# Visualizziamo i risultati
rows = []
for cid, data in cluster_labels.items():
    rows.append({
        "Cluster ID": cid,
        "LLM Name": data.get("llm_summary", ""),
        "Top Keywords (Input)": ", ".join(data.get("ctfidf", [])[:5])
    })

df_results = pd.DataFrame(rows)
display(df_results)

,Cluster ID,LLM Name,Top Keywords (Input)
0,-1,DB Wealth,"deutsche, td, bank, communication, com"
1,0,NY Times,"times, newyorktimesinfo, com, new, nytimes"
2,1,Gonzales Statement,"wikisource, senator, alberto, statement, gonzales"
3,2,Case Assignment,"case, assigned, salesforce, assignment, notifi..."
4,3,KYC Clearance,"alert, compliance, pcr, jeffrey, epstein"
5,4,Wanek Holdings,"llc, 00, td, grat, wanek"
6,5,Lexington Associates,"lexington, 4th, meeting, 10022, associates"
7,6,Financial Services,"financial, southern, llc, edi, 127608"
8,7,Financial Compliance,"edd, package, southern, dmitri, rid"
9,8,Legal Document,"confidential, fed, crim, pursuant, db"
